In [0]:
# Pass your Personal Access Token (PAT) here
# Scope required on GitHub: read:user or fine-grained public repositories read-only
GITHUB_TOKEN = dbutils.secrets.get(
    scope="pipeline-signal",
    key="github-token"
)

headers = {"User-Agent": "Mozilla/5.0"}
if GITHUB_TOKEN:
    headers["Authorization"] = f"bearer {GITHUB_TOKEN}"
    print("✅ GitHub Token set!!! API limit raised to 5,000 requests/hr.")
else:
    print("⚠️ No token set. Running on public rate limit (60 requests/hr).")

In [0]:
import requests
import json
import base64
from pyspark.sql.functions import current_timestamp

CATALOG = "pipeline_signal"
SCHEMA = "bronze"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

def save_to_bronze(df, table_name):
    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .option("delta.enableChangeDataFeed", "true") \
      .saveAsTable(full_table)
    print(f"✅ Created Bronze Table: `{full_table}` (Total Rows: {df.count()})")


# ---------------------------------------------------------
# 1. BRONZE: metadata_mce (Full DataHub Bootstrap MCE)
# ---------------------------------------------------------
print("--- [1/4] Ingesting Full DataHub MCE ---")
mce_url = "https://raw.githubusercontent.com/datahub-project/datahub/master/metadata-ingestion/examples/mce_files/bootstrap_mce.json"
mce_resp = requests.get(mce_url)
if mce_resp.status_code == 200:
    mce_data = mce_resp.json()
    mce_records = [{"raw_event": json.dumps(event)} for event in mce_data]
    df_mce = spark.createDataFrame(mce_records).withColumn("ingested_at", current_timestamp())
    save_to_bronze(df_mce, "metadata_mce")


# ---------------------------------------------------------
# 2. BRONZE: dbt_manifest (Full dbt Manifest Node Tree)
# ---------------------------------------------------------
print("\n--- [2/4] Ingesting Complete dbt Manifest Artifacts ---")
manifest_url = "https://raw.githubusercontent.com/dbt-labs/dbt-core/main/tests/functional/artifacts/expected_manifest.json"
manifest_resp = requests.get(manifest_url)
if manifest_resp.status_code == 200:
    manifest_data = manifest_resp.json()
    nodes = manifest_data.get("nodes", {})
    manifest_records = [
        {
            "node_id": k,
            "resource_type": v.get("resource_type", "unknown"),
            "raw_code": v.get("raw_code", v.get("compiled_code", "")),
            "raw_node": json.dumps(v)
        }
        for k, v in nodes.items()
    ]
    df_manifest = spark.createDataFrame(manifest_records).withColumn("ingested_at", current_timestamp())
    save_to_bronze(df_manifest, "dbt_manifest")


# ---------------------------------------------------------
# 3. BRONZE: github_issues (Paging to Extract All Available Issues)
# ---------------------------------------------------------
print("\n--- [3/4] Ingesting Bulk DataHub GitHub Issues ---")
issue_records = []
page = 1

while True:
    issues_url = f"https://api.github.com/repos/datahub-project/datahub/issues?state=all&per_page=100&page={page}"
    resp = requests.get(issues_url, headers=headers)
    
    if resp.status_code != 200:
        print(f"⚠️ Stopped paging issues at page {page}. Status: {resp.status_code}")
        break
        
    data = resp.json()
    if not data or not isinstance(data, list):
        break
        
    for issue in data:
        if "pull_request" not in issue:
            issue_records.append({
                "issue_id": str(issue.get("id")),
                "number": issue.get("number"),
                "title": issue.get("title", ""),
                "body": issue.get("body", "") or "",
                "state": issue.get("state", ""),
                "html_url": issue.get("html_url", ""),
                "raw_issue": json.dumps(issue)
            })
            
    # Stop condition if we reach 5 pages without token (to avoid immediate IP lock) or end of data
    if len(data) < 100 or (not GITHUB_TOKEN and page >= 5):
        break
    page += 1

if issue_records:
    df_issues = spark.createDataFrame(issue_records).withColumn("ingested_at", current_timestamp())
    save_to_bronze(df_issues, "github_issues")


# ---------------------------------------------------------
# 4. BRONZE: documentation (Full Recursive Repo Scraping)
# ---------------------------------------------------------
print("\n--- [4/4] Ingesting Complete Documentation Tree ---")

def fetch_all_docs(repo_path="docs"):
    records = []
    api_url = f"https://api.github.com/repos/datahub-project/datahub/contents/{repo_path}"
    resp = requests.get(api_url, headers=headers)
    
    if resp.status_code == 200:
        items = resp.json()
        for item in items:
            if isinstance(item, dict):
                if item.get("type") == "dir":
                    records.extend(fetch_all_docs(item.get("path")))
                elif item.get("name", "").endswith(".md"):
                    file_resp = requests.get(item["url"], headers=headers)
                    if file_resp.status_code == 200:
                        content_json = file_resp.json()
                        if "content" in content_json:
                            raw_text = base64.b64decode(content_json["content"]).decode("utf-8", errors="ignore")
                            records.append({
                                "doc_id": item.get("sha", item["name"]),
                                "path": item.get("path", ""),
                                "title": item.get("name", "").replace(".md", "").replace("-", " ").title(),
                                "category": repo_path,
                                "content": raw_text,
                                "html_url": item.get("html_url", "")
                            })
    return records

all_docs = fetch_all_docs("docs")
if all_docs:
    df_docs = spark.createDataFrame(all_docs).withColumn("ingested_at", current_timestamp())
    save_to_bronze(df_docs, "documentation")

print("\n🎉 BULK INGESTION COMPLETE ACROSS ALL BRONZE TABLES!")

In [0]:
import json
from pyspark.sql.functions import current_timestamp

CATALOG = "pipeline_signal"
SCHEMA = "bronze"

# Standard dbt manifest JSON payload
manifest_json_str = """
{
  "metadata": {
    "dbt_schema_version": "https://schemas.getdbt.com/dbt/manifest/v12.json",
    "dbt_version": "1.9.0",
    "project_name": "tdd",
    "adapter_type": "athena"
  },
  "nodes": {
    "model.tdd.simple": {
      "database": "awsdatacatalog",
      "schema": "sandbox",
      "name": "simple",
      "resource_type": "model",
      "package_name": "tdd",
      "path": "simple.sql",
      "original_file_path": "models/simple.sql",
      "unique_id": "model.tdd.simple",
      "description": "This model calculates the count of records in the stg_simple table.",
      "columns": {
        "action_type_id": {
          "name": "action_type_id",
          "description": "The count of records in the stg_simple table",
          "data_type": "integer"
        },
        "mixed": {
          "name": "mixed",
          "description": "",
          "data_type": "varchar"
        }
      },
      "relation_name": "\\"awsdatacatalog\\".\\"sandbox\\".\\"simple\\"",
      "raw_code": "select 1 as action_type_id, 2 as mixed from test.test",
      "language": "sql",
      "depends_on": {
        "nodes": [
          "model.tdd.stg_simple"
        ]
      }
    }
  }
}
"""

manifest_data = json.loads(manifest_json_str)
nodes = manifest_data.get("nodes", {})

manifest_records = [
    {
        "node_id": k,
        "resource_type": v.get("resource_type", "unknown"),
        "raw_code": v.get("raw_code", v.get("compiled_code", "")),
        "raw_node": json.dumps(v)
    }
    for k, v in nodes.items()
]

df_manifest = spark.createDataFrame(manifest_records).withColumn("ingested_at", current_timestamp())

df_manifest.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .option("delta.enableChangeDataFeed", "true") \
  .saveAsTable(f"{CATALOG}.{SCHEMA}.dbt_manifest")

print(f"✅ Created Bronze Table: `{CATALOG}.{SCHEMA}.dbt_manifest` (Total Rows: {df_manifest.count()})")